# Chapter 9: Test Case Generation

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build a **Test Case Generator** that takes requirements or user stories and produces comprehensive test cases covering functional, boundary, negative, and edge cases.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Understanding of test case design techniques


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Requirements-Based Test Case Generation

Generate test cases from a detailed requirement, covering all test types.


In [ ]:
# Define a requirement to test
requirement = {
    'id': 'REQ-042',
    'title': 'Password Validation',
    'description': """The system shall enforce the following password policy:
    - Minimum 8 characters, maximum 64 characters
    - Must contain at least one uppercase letter (A-Z)
    - Must contain at least one lowercase letter (a-z)
    - Must contain at least one digit (0-9)
    - Must contain at least one special character (!@#$%^&*)
    - Cannot contain the user's email address or username
    - Cannot be any of the last 5 passwords used
    - Password strength meter must update in real-time as user types
    - Show/hide password toggle must be available"""
}

# Generate comprehensive test cases
test_prompt = f"""Generate comprehensive test cases for this requirement.

Requirement: {json.dumps(requirement)}

Generate test cases in these categories:
1. Positive/Happy Path (valid inputs)
2. Negative (invalid inputs that should be rejected)
3. Boundary Value (at the exact limits)
4. Edge Cases (unusual but valid scenarios)
5. Security (attack vectors)

For each test case provide:
- tc_id: TC-042-xxx format
- title: descriptive title
- category: one of the 5 above
- preconditions: setup needed
- steps: numbered list of actions
- test_data: specific input values to use
- expected_result: what should happen
- priority: Critical/High/Medium/Low

Generate at least 15 test cases. Return as a JSON array. Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a senior QA engineer specializing in test design.'},
        {'role': 'user', 'content': test_prompt}
    ],
    temperature=0.2
)
test_cases = json.loads(response.choices[0].message.content)
print(f'Generated {len(test_cases)} test cases\n')

# Summary by category
from collections import Counter
categories = Counter(tc['category'] for tc in test_cases)
for cat, count in categories.most_common():
    print(f'  {cat}: {count} test cases')

## 2. Display and Explore Test Cases

View the generated test cases in detail.


In [ ]:
# Display test cases by category
for category in ['Positive/Happy Path', 'Negative', 'Boundary Value', 'Edge Cases', 'Security']:
    matching = [tc for tc in test_cases if category.lower() in tc['category'].lower()]
    if matching:
        print(f"\n{'='*60}")
        print(f'{category} ({len(matching)} tests)')
        print(f"{'='*60}")
        for tc in matching[:3]:  # Show first 3 per category
            print(f"\n{tc['tc_id']}: {tc['title']}")
            print(f"  Priority: {tc['priority']}")
            print(f"  Test Data: {tc.get('test_data', 'N/A')}")
            print(f"  Expected: {tc['expected_result'][:100]}")

In [ ]:
# Coverage analysis: check if test cases cover all requirement clauses
coverage_prompt = f"""Analyze test coverage for this requirement.

Requirement: {json.dumps(requirement)}
Test Cases: {json.dumps(test_cases)}

For each clause in the requirement:
1. List the clause
2. List which test case IDs cover it
3. Identify if coverage is sufficient (fully covered, partially covered, not covered)
4. Suggest additional test cases for any gaps

Return a JSON object with:
- clauses: array of {{clause, covering_tests, coverage_status, suggested_additions}}
- overall_coverage_pct: estimated percentage
- gaps: list of untested scenarios

Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': coverage_prompt}],
    temperature=0.2
)
coverage = json.loads(response.choices[0].message.content)
print(f"Overall Coverage: {coverage['overall_coverage_pct']}%\n")
for clause in coverage['clauses']:
    status_icon = {'fully covered': '[OK]', 'partially covered': '[!!]', 'not covered': '[XX]'}
    icon = status_icon.get(clause['coverage_status'].lower(), '[??]')
    print(f"{icon} {clause['clause'][:70]}")
    print(f"     Tests: {', '.join(clause['covering_tests']) if clause['covering_tests'] else 'NONE'}")
    if clause.get('suggested_additions'):
        print(f"     Suggestion: {clause['suggested_additions']}")

## Exercises
1. Generate test cases for an API endpoint (e.g., POST /api/users) including status codes.
2. Add a test case prioritization step that orders tests by risk and business impact.
3. Export the test cases to CSV format compatible with your test management tool.
4. Create a test case reviewer that checks for redundant or overlapping test cases.
